# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [ ]:
df = con.sql(f"""
WITH content_id_hv_criteria AS (
    SELECT
        content_hash_id
    FROM {TABLES['fact_content_daily_performance']}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS NOT NULL
    GROUP BY content_hash_id
    HAVING
        COUNT(DISTINCT report_date) >= 120
        AND
        DATE_DIFF(
            'day',
            MIN(report_date),
            MAX(report_date)
        ) + 1 = COUNT(DISTINCT report_date)
)

SELECT
      report_date,
      client_hash_id,
      fact.content_hash_id,
      ga4_data_available,
      gsc_data_available,
      gsc_impressions,
      gsc_clicks,
      gsc_sum_position,
      gsc_avg_position,
      ga4_pageviews,
      ga4_sessions,
      ga4_users,
      ga4_engaged_sessions,
      sessions_ai,
      sessions_organic,
      scroll_events

FROM {TABLES['fact_content_daily_performance']} AS fact
INNER JOIN content_id_hv_criteria AS criteria
    ON fact.content_hash_id = criteria.content_hash_id
WHERE
    fact.gsc_data_available IS TRUE
    AND fact.ga4_data_available IS NOT NULL;
""").df()

df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
result = df.groupby('content_hash_id').agg(start_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['start_date','end_date'], ascending=[False,True])
result

In [ ]:
import pandas as pd

In [ ]:
incomplete_content_month = result.copy().reset_index()

incomplete_content_month['incomplete_start_month'] = (
    incomplete_content_month['start_date'].dt.day > 15
)

incomplete_content_month['incomplete_end_month'] = (
    incomplete_content_month['end_date'].dt.day < 15
)

incomplete_content_month['start_month'] = (
    incomplete_content_month['start_date'].dt.to_period('M')
)

incomplete_content_month['end_month'] = (
    incomplete_content_month['end_date'].dt.to_period('M')
)

df['month'] = df['report_date'].dt.to_period('M')

data_complete_month = df.merge(
    incomplete_content_month[
        [
            'content_hash_id',
            'start_month',
            'end_month',
            'incomplete_start_month',
            'incomplete_end_month'
        ]
    ],
    on='content_hash_id',
    how='left'
)

data_complete_month = data_complete_month[
    ~(
        (
            data_complete_month['incomplete_start_month'] &
            (data_complete_month['month'] == data_complete_month['start_month'])
        ) |
        (
            data_complete_month['incomplete_end_month'] &
            (data_complete_month['month'] == data_complete_month['end_month'])
        )
    )
].copy()

In [ ]:
data_complete_month.groupby('content_hash_id').agg(days = ('report_date', "count")).sort_values('days',ascending=True)

In [ ]:
data_complete_month[data_complete_month['content_hash_id'] == 'content_f3ba8c1f94eb36f5']

In [ ]:
data_complete_month[(data_complete_month['gsc_avg_position']<1)]

In [ ]:
data_complete_month.drop(['start_month', 'end_month','incomplete_start_month','incomplete_end_month'], axis=1, inplace=True)

In [ ]:
grouped = data_complete_month.groupby([ 'client_hash_id','content_hash_id', 'month'])

numeric_cols = data_complete_month.select_dtypes(include='number').columns.drop('gsc_avg_position')
bool_cols = data_complete_month.select_dtypes(include='bool').columns

content_month = grouped.agg({
    **{col: 'sum' for col in numeric_cols},
    **{col: 'any' for col in bool_cols},
    'gsc_avg_position':'mean'
}).reset_index()

rename_dict = {
    col: f"{col}_30d" for col in numeric_cols.append(pd.Index(['gsc_avg_position']))
}
content_month = content_month.rename(columns=rename_dict)

In [ ]:
content_month

In [ ]:
content_month.info()

In [ ]:
cols = content_month.columns.drop(['client_hash_id', 'content_hash_id', 'month', 'gsc_data_available', 'ga4_data_available'])
cols

In [ ]:
content_month = content_month.sort_values(
    ['client_hash_id', 'content_hash_id', 'month']
).copy()

grouped = content_month.groupby(
    ['client_hash_id', 'content_hash_id']
)

for col in cols:
    base_col = col[:-4]

    content_month[f'{base_col}_prev_30d'] = grouped[col].shift(1)
    content_month[f'{base_col}_2m_ago_30d'] = grouped[col].shift(2)
    content_month[f'{base_col}_future_30d'] = grouped[col].shift(-1)

In [ ]:
content_month[content_month['content_hash_id'] == 'content_f3baf9504688']

In [ ]:
dataset = content_month[
    content_month['gsc_impressions_2m_ago_30d'].notna() &
    content_month['gsc_impressions_future_30d'].notna()
].copy()

In [ ]:
dataset.info()

In [ ]:
contents = con.sql(f"""
SELECT
    content_hash_id,
    content_created_date,
    content_type,
    search_volume,
    competition,
    competition_level,
    cpc,
    main_intent,
    backlinks,
    category_count,
    word_count,
    char_count
FROM {TABLES['dim_content']}
""").df()

contents

In [ ]:
contents.info()

In [ ]:
dataset = dataset.merge(contents, on='content_hash_id', how='inner')
dataset

In [ ]:
dataset['prediction_date'] = (
    pd.to_datetime(dataset['month'].dt.end_time.dt.date)
)

dataset['content_age_days'] = (
    dataset['prediction_date'] - dataset['content_created_date']
).dt.days


In [ ]:
impoer numpy as np

dataset['ctr_30d'] = (dataset['gsc_clicks_30d'] / dataset['gsc_impressions_30d']) * 100
dataset['engagement_rate_30d'] = (dataset['ga4_engaged_sessions_30d'] / dataset['ga4_sessions_30d']) * 100
dataset['scroll_rate_30d'] = (dataset['scroll_events_30d'] / dataset['ga4_pageviews_30d']) * 100
dataset['ai_traffic_30d'] = (dataset['sessions_ai_30d'] / dataset['ga4_sessions_30d']) * 100
dataset['trend_pct_30d'] = ((dataset['gsc_impressions_30d']  - dataset['gsc_impressions_prev_30d']) / dataset['gsc_impressions_prev_30d']) * 100
dataset['trend_pct_future'] = ((dataset['gsc_impressions_future_30d']  - dataset['gsc_impressions_30d']) / dataset['gsc_impressions_30d']) * 100
dataset['decline_label'] = (dataset['trend_pct_30d'] < -20).astype(int)
dataset['decline_future_label'] = (dataset['trend_pct_future'] < -20).astype(int)
dataset['position_tier_30d'] = pd.cut(
    dataset["gsc_avg_position_30d"],
    bins=[
        99,
        249,
        499,
        999,
        np.inf
    ],
    labels=[
        "100-249",
        "250-499",
        "500-999",
        "1000+"
    ],
    include_lowest=True
)




In [ ]:
dataset.info()

In [ ]:
dataset[(dataset['gsc_avg_position_30d']<1)]['gsc_avg_position_30d']

In [ ]:
dataset[(dataset['content_age_days']<1)]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
corr = dataset[dataset.select_dtypes(include=['number']).columns].corr()

fig, ax = plt.subplots(figsize=(35, 35))
sns.heatmap(corr, annot = True, cmap='coolwarm')

In [ ]:
dataset_imp_500 = dataset[dataset['trend_pct_30d'] < -20].copy()

In [ ]:
dataset_imp_500

In [ ]:
dataset_imp_500.plot.scatter(x='gsc_avg_position_30d', y='gsc_clicks_30d', title='Height vs. Weight')
plt.show()

In [ ]:
dataset['trend_pct_future'] < -2

In [ ]:
dataset[dataset['trend_pct_future'] < -20]

In [ ]:
dataset.to_csv("dataset_flyrank")

In [ ]:
feature_cols = [
    'ga4_data_available',
    'gsc_data_available'
    'gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_users',
    'ga4_engaged_sessions',
    'sessions_ai',
    'sessions_organic',
    'scroll_events',

    'content_created_date',
    'content_updated_date',
    'content_type',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'main_intent',
    'backlinks',
    'category_count',
    'word_count',
    'char_count',
]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.